#### **The Keras functional API**

This notebook covers Keras callbacks and the TensorBoard browser-based visualization tool let you monitor models during training. We will also discuss several other best practices including bacth normalization, residual connections, hyperparameter optimization, and model ensembling.

#### **Going beyond the sequential model**

Similarly, some tasks need to predict multiple target attributes of input data. Many  nueral architectutes (Inception, from Google or ResNet from Microsoft) require nonlinear network topology: networks structured as a directed acyclic graphs. These scenarios (multi-input multi-ouput models, graph like models) aren't possible  when using only the Sequential model class in Keras. So let's use the keras functional API.

#### **Introduction to functional API**

In [1]:
from keras import Input, layers

input_tensor = Input((32,)) # a tensor
dense = layers.Dense(32, activation='relu') # a layer is a function
output_tensor = dense(input_tensor) # a layer may be called on a tensor, and it returns a tensor


Let's start with a minimal example that shows side by side a simple sequential model and its equivalent  in the functional API.

In [2]:
from keras.models import Sequential, Model
from keras import layers
from keras import Input

seq_model = Sequential()
seq_model.add(layers.Dense(32, activation='relu', input_shape=(64,)))
seq_model.add(layers.Dense(32, activation='relu'))
seq_model.add(layers.Dense(10, activation='softmax'))

# Functional equivalent of sequential
input_tensor = Input(shape=(64,))
x = layers.Dense(32, activation='relu')(input_tensor)
x = layers.Dense(32, activation = 'relu')(x)
output_tensor = layers.Dense(10, activation='softmax')(x)

model = Model(input_tensor, output_tensor) # The model class turns an input tensor and output tensor into a model
model.summary() # Let's look at it.

model.compile(optimizer='rmsprop', loss='categorical_crossentropy') # compiles the model

# Genertates random numpy data to trained on
import numpy as np
x_train = np.random.random((1000, 64))
y_train = np.random.random((1000,10))

model.fit(x_train, y_train, epochs=10, batch_size=128) # Trains the model for 10 epochs
score = model.evaluate(x_train, y_train)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,466 (13.54 KB)

 Trainable params: 3,466 (13.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 107ms/step - loss: 12.0558
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 12.5080 
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 14.2369 
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 17.4600 
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 21.7137 
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 26.2354 
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 31.3309 
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 36.5014 
Epoch 9/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 42.3944 
Epoch 10/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 48.8203 
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 52.5172


#### **Multi-input models**

Typically these models at some point merge their different input branches using a layer that can combine several tensors. Let's look at a very simple example of a multi-input model: a question-answering model.

How we can build such a model with the functional API. We setup two independent branches, encoding the text input and the question input as representation vectors: then, concatenate these vectors; and finally add a softmax classifier on top of the concatenated representations.

In [3]:
### Functional API implementation of a two-input question-answering model

from keras.models import Model
from keras import layers
from keras import Input

text_vocabulary_size = 10000
question_vocabulary_size = 10000
answer_vocabulary_size = 500

# The text input is a variable-length sequence of integers.
# Note that we can optionally name the inputs
text_input = Input(shape=(None,), dtype='int32', name='text')

embedded_text = layers.Embedding(64, text_vocabulary_size)(text_input) # Embeds the inputs into a sequence of vectors of size 64
encoded__text = layers.LSTM(32)(embedded_text) # Encodes the vectors in a single vector via an LSTM

# For question
question_input = Input(shape=(None,), dtype='int32', name='question')
embedded_question = layers.Embedding(32, question_vocabulary_size)(question_input)
encoded_question = layers.LSTM(16)(embedded_question)

# Concatenate the encoded text and encoded question
merged = layers.concatenate([encoded__text, encoded_question], axis=-1)

# And we add a softmax classifier on top
answer = layers.Dense(answer_vocabulary_size, activation='softmax')(merged) # Adds a softmax classifier on top

# This defines our model
model = Model([text_input, question_input], answer) # At model instantiation, specify the two inputs and the output.

model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None, None)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ question            │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None,      │    640,000 │ text[0][0]        │
│ (Embedding)         │ 10000)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None,      │    320,000 │ question[0][0]    │
│ (Embedding)         │ 10000)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 32)        │  1,284,224 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 16)        │    641,088 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 48)        │          0 │ lstm[0][0],       │
│ (Concatenate)       │                   │            │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 500)       │     24,500 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,909,812 (11.10 MB)

 Trainable params: 2,909,812 (11.10 MB)

 Non-trainable params: 0 (0.00 B)

Now, how to train this two-input model? There are two possible APIs: you can feed the model a list of Numpy arrays as inputs, or you can feed it a dictionary that maps input names to Numpy arrays.

In [4]:
import numpy as np

num_samples = 1000
max_length = 100

text = np.random.randint(1, text_vocabulary_size, size=(num_samples, max_length))
question = np.random.randint(1, question_vocabulary_size, size=(num_samples, max_length))

# Note that we cannot pass a numpy array of integers for the target,
#we must one-hot encode it.
# Our targets will be one-hot encoded vectors
answers = np.random.randint(0, 1, size=(num_samples, answer_vocabulary_size)).astype('float32') # dummy answers

# Configure the model with an optimizer and a loss
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['acc'])

# Train the model with list of inputs.
# Note that since the names of the inputs were passed during the creation of the model
# we can pass either a dictionary or a list of numpy arrays.
# If you choose to pass a list, then the arrays in the list must correspond to the order
# of the inputs defined in the model. So, in this case, the first element is 'text_input'
# and the second is 'question_input'.
model.fit([text, question], answers, epochs=10, batch_size=128)

Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 8s 146ms/step - acc: 0.0720 - loss: 0.0000e+00
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 145ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 138ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 140ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 9/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 140ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 10/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 140ms/step - acc: 0.0000e+00 - loss: 0.0000e+00


In [5]:
# using dictionary
model.fit({'text': text, 'question': question}, answers, epochs=10, batch_size=128)

Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 144ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 144ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 141ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 9/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 144ms/step - acc: 0.0000e+00 - loss: 0.0000e+00
Epoch 10/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 145ms/step - acc: 0.0000e+00 - loss: 0.0000e+00


#### **Multi-Output Models**

In the same way, we can use the functional API to build models with multiple outputs

In [6]:
### Functional API implementation of a three-output model

from keras import layers
from keras import Input
from keras.models import Model

vocabulary_size = 5000
num_income_groups = 10

posts_input =Input(shape=(None,), dtype='int32', name='posts')
# Corrected Embedding layer: input_dim should be vocabulary_size, output_dim should be 256
embedded_posts = layers.Embedding(vocabulary_size, 256)(posts_input)

x = layers.Conv1D(128, 5, activation='relu', padding="same")(embedded_posts)
x = layers.MaxPooling1D(5)(x)
x = layers.Conv1D(256, 5, activation='relu', padding="same")(x)
x = layers.Conv1D(256, 5, activation='relu', padding="same")(x)
x = layers.MaxPooling1D(2)(x) # Reduced pool_size from 5 to 2
x = layers.Conv1D(256, 3, activation='relu', padding="same")(x) # Reduced kernel_size from 5 to 3
x = layers.Conv1D(256, 3, activation='relu', padding="same")(x) # Reduced kernel_size from 5 to 3
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dense(128, activation='relu')(x)

age_prediction = layers.Dense(1, name='age')(x) # Note that the output layers are given names
income_prediction = layers.Dense(num_income_groups,
                                 activation='softmax',
                                 name='income')(x)
gender_prediction = layers.Dense(1, activation='sigmoid', name='gender')(x)
model = Model(posts_input, [age_prediction, income_prediction, gender_prediction])

model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ posts (InputLayer)  │ (None, None)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 256) │  1,280,000 │ posts[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, None, 128) │    163,968 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, None, 128) │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, None, 256) │    164,096 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, None, 256) │    327,936 │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, None, 256) │          0 │ conv1d_2[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, None, 256) │    196,864 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, None, 256) │    196,864 │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 256)       │          0 │ conv1d_4[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 128)       │     32,896 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ age (Dense)         │ (None, 1)         │        129 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ income (Dense)      │ (None, 10)        │      1,290 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender (Dense)      │ (None, 1)         │        129 │ dense_8[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,364,172 (9.02 MB)

 Trainable params: 2,364,172 (9.02 MB)

 Non-trainable params: 0 (0.00 B)

Importantly, training such a model requires the ability to specify different loss functions for different heads of the network: for instance, age prediction is a scalar regression task, but gender prediction is a binary classification task, requiring a different procedure. But because gradient descent requires us to minimize a scalar, we must combine these losses into a single value in order to train the model. We can sum them all.

In [7]:
### Compilation options of a multi-output model: multiple losses
model.compile(optimizer='rmsprop',
              loss={'age':'mse',
                    'income': 'categorical_crossentropy',
                    'gender':'binary_crossentropy'})

In [8]:
### Feeding data to a multi-output model

import numpy as np

num_samples = 5000
num_income_groups = 10

# Also using vocabulary_size and max_length from previous cells (CvZSw5sPcx8x, oUKhIEfUbvwg)
vocabulary_size = 5000
max_length = 100

# Create dummy data for the posts input
posts_data = np.random.randint(1, vocabulary_size, size=(num_samples, max_length)).astype('int32')

age_targets = np.random.rand(num_samples, 1) * 100 # Dummy age targets, e.g., 0-100
income_targets = np.random.randint(0, 2, size=(num_samples, num_income_groups)).astype('float32') # Dummy one-hot encoded income targets
gender_targets = np.random.randint(0, 2, size=(num_samples, 1)).astype('float32') # Dummy binary gender targets

model.fit(posts_data, {
    'age': age_targets,
    'income': income_targets,
    'gender': gender_targets
}, epochs=10, batch_size=64)

Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 23s 90ms/step - age_loss: 1225.7770 - gender_loss: 0.7970 - income_loss: 29.1089 - loss: 1256.6478
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - age_loss: 6452.1528 - gender_loss: 7.4635 - income_loss: 410.1626 - loss: 6921.1680
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - age_loss: 65186.5742 - gender_loss: 31.4248 - income_loss: 1218.4529 - loss: 67155.7422
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - age_loss: 123181.3281 - gender_loss: 43.0069 - income_loss: 1027.5901 - loss: 125623.7188
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - age_loss: 170772.6406 - gender_loss: 54.1754 - income_loss: 890.7015 - loss: 171590.7812
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - age_loss: 392825.0625 - gender_loss: 88.5879 - income_loss: 1262.6298 - loss: 398576.2188
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - age_loss: 1099360.0000 - gender_loss: 147.6959 - income_loss: 2264.3196 - loss: 803576.5000
Epoch 8/10
79/79

Just summing creates a very imbalanced loss contributions. We can  also control the loss weights.

In [9]:
### Compilation options of a multi-output model: loss weighting

model.compile(optimizer='rmsprop',
              loss={'age':'mse',
                    'income': 'categorical_crossentropy',
                    'gender': 'binary_crossentropy'},
              loss_weights={'age': 0.25,
                            'income': 1.0,
                            'gender': 10.00})

In [10]:
model.fit(posts_data, {
    'age': age_targets,
    'income': income_targets,
    'gender': gender_targets
}, epochs=10, batch_size=64)

Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 16s 88ms/step - age_loss: 25453354.0000 - gender_loss: 704.9737 - income_loss: 13320.5820 - loss: 5960014.0000
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - age_loss: 49732448.0000 - gender_loss: 1316.8737 - income_loss: 22857.3027 - loss: 12606403.0000
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - age_loss: 55578112.0000 - gender_loss: 1317.3105 - income_loss: 25215.2715 - loss: 14088548.0000
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - age_loss: 69778208.0000 - gender_loss: 1645.6537 - income_loss: 28284.6523 - loss: 17684950.0000
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - age_loss: 134126504.0000 - gender_loss: 2004.3362 - income_loss: 42729.7578 - loss: 33970452.0000
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - age_loss: 122781880.0000 - gender_loss: 2160.9844 - income_loss: 40255.9219 - loss: 31101312.0000
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - age_loss: 162540096.0000 - gender_loss: 2577.6038 -

#### **Directed Acyclic graphs of layers**

We also implement netowrks with complex internal topology. We should keep in mind that these graphs can't have cycles. The only processing loops that are allowed are those internal to recurrent layers.

**Spoiler Alert:** We are going to implement the inception and residual connections.

##### **Inception Modules**

Inception (https://arxiv.org/abs/1409.4842) consists of stack of modules that themselves look like small independent networks, split into several parallel branches.

The most basic form of an Inception module has three to four branches strating with a 1 x 1 convolution, followed by a 3 x 3 convolution, and ending with a concatenation of the resulting features. This setup helps the network separately learn spatial features and channel-wise features which is more efficient than learning them jointly.

In [11]:
### building inception module for MNIST digit classification

from keras import layers
from keras import Input
from keras.models import Model
import numpy as np
from keras.utils import to_categorical
from keras.datasets import mnist # Import MNIST dataset

# For MNIST, input is 28x28 grayscale images
mnist_input = Input(shape=(28, 28, 1), dtype='float32', name='mnist_image')

# Initial convolution (equivalent to the previous 'x' layer, adapted for 2D)
x = layers.Conv2D(128, (5, 5), activation='relu', padding="same")(mnist_input)

#### Every branch has the same stride value (2), which is necessary to keep all branch outputs the same size so you can concatenate them
# Branch A: 1x1 convolution with stride 2
branch_a = layers.Conv2D(128, (1, 1), activation='relu', strides=(2, 2), padding="same")(x);

# Branch B: 1x1 convolution followed by 3x3 convolution with stride 2
branch_b = layers.Conv2D(128, (1, 1), activation='relu', padding="same")(x);
branch_b = layers.Conv2D(128, (3, 3), activation='relu', strides=(2, 2), padding="same")(branch_b);

# Branch C: Average pooling with stride 2, followed by 3x3 convolution
branch_c = layers.AveragePooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x);
branch_c = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(branch_c); # Corrected layers.conv2d to layers.Conv2D

# Branch D: 1x1 convolution, then 3x3, then another 3x3 convolution with stride 2
branch_d = layers.Conv2D(128, (1, 1), activation='relu', padding='same')(x);
branch_d = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(branch_d);
branch_d = layers.Conv2D(128, (3, 3), activation='relu', strides=(2, 2), padding='same')(branch_d);

# Concatenate the branch outputs to obtain the module output
# All branches should now have output shape (batch, 14, 14, 128) - after stride 2 on 28x28 input to branches
output = layers.concatenate([branch_a, branch_b, branch_c, branch_d], axis=-1)

# Add a classification head for MNIST
output = layers.GlobalAveragePooling2D()(output)
mnist_output = layers.Dense(10, activation='softmax', name='digits')(output)

# Define the model
model = Model(mnist_input, mnist_output)

model.summary()

# --- Load and preprocess actual MNIST data ---
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Reshape and normalize images
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1)).astype('float32') / 255
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1)).astype('float32') / 255

# One-hot encode labels
y_train_one_hot = to_categorical(y_train, num_classes=10)
y_test_one_hot = to_categorical(y_test, num_classes=10)

# Compile the model
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model with actual data
model.fit(x_train, y_train_one_hot, epochs=5, batch_size=64, validation_data=(x_test, y_test_one_hot))

# Evaluate the model on the test set
loss, accuracy = model.evaluate(x_test, y_test_one_hot)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mnist_image         │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 28, 28,    │      3,328 │ mnist_image[0][0] │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 28, 28,    │     16,512 │ conv2d[0][0]      │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 28, 28,    │     16,512 │ conv2d[0][0]      │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ average_pooling2d   │ (None, 14, 14,    │          0 │ conv2d[0][0]      │
│ (AveragePooling2D)  │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 28, 28,    │    147,584 │ conv2d_5[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 14, 14,    │     16,512 │ conv2d[0][0]      │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 14, 14,    │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 14, 14,    │    147,584 │ average_pooling2… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 14, 14,    │    147,584 │ conv2d_6[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 14, 14,    │          0 │ conv2d_1[0][0],   │
│ (Concatenate)       │ 512)              │            │ conv2d_3[0][0],   │
│                     │                   │            │ conv2d_4[0][0],   │
│                     │                   │            │ conv2d_7[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ concatenate_1[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ digits (Dense)      │ (None, 10)        │      5,130 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 648,330 (2.47 MB)

 Trainable params: 648,330 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 46s 43ms/step - accuracy: 0.5759 - loss: 1.2054 - val_accuracy: 0.8691 - val_loss: 0.4339
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 32s 34ms/step - accuracy: 0.8917 - loss: 0.3668 - val_accuracy: 0.9193 - val_loss: 0.2831
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 33s 35ms/step - accuracy: 0.9454 - loss: 0.1837 - val_accuracy: 0.9290 - val_loss: 0.2084
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 41s 35ms/step - accuracy: 0.9620 - loss: 0.1253 - val_accuracy: 0.9228 - val_loss: 0.2648
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 32s 34ms/step - accuracy: 0.9702 - loss: 0.0985 - val_accuracy: 0.9066 - val_loss: 0.3060
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9066 - loss: 0.3060
Test Loss: 0.3060
Test Accuracy: 0.9066


We have successfully build a inception connections. Now let's build the residual connections.

##### **Residual Connections**

They tackle two common problems that plague any large scale deep learning model: vanishing gradients and representational bottlenecks.

A residual connection consists of making the output of an earlier layer available as input to a later layer, effectively creating a shortcut in a sequential network.

In [12]:
### Implementation of a residual connection

x = layers.Conv2D(128, (5, 5), activation='relu', padding="same")(mnist_input)
y = layers.Conv2D(128, 3, activation='relu', padding='same')(x) # Applies a transformation to x
y = layers.Conv2D(128, 3, activation='relu', padding='same')(y)
y = layers.Conv2D(128, 3, activation='relu', padding='same')(y)

y = layers.add([y, x]) # Adds the original x back to the output features

# Add a classification head for MNIST
output = layers.GlobalAveragePooling2D()(y)
mnist_output = layers.Dense(10, activation='softmax', name='digits')(output)

# Define the model
model = Model(mnist_input, mnist_output)

model.summary()

# --- Load and preprocess actual MNIST data ---
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Reshape and normalize images
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1)).astype('float32') / 255
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1)).astype('float32') / 255

# One-hot encode labels
y_train_one_hot = to_categorical(y_train, num_classes=10)
y_test_one_hot = to_categorical(y_test, num_classes=10)

# Compile the model
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model with actual data
model.fit(x_train, y_train_one_hot, epochs=5, batch_size=64, validation_data=(x_test, y_test_one_hot))

# Evaluate the model on the test set
loss, accuracy = model.evaluate(x_test, y_test_one_hot)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mnist_image         │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 28, 28,    │      3,328 │ mnist_image[0][0] │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 28, 28,    │    147,584 │ conv2d_8[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 28, 28,    │    147,584 │ conv2d_9[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 28, 28,    │    147,584 │ conv2d_10[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 28, 28,    │          0 │ conv2d_11[0][0],  │
│                     │ 128)              │            │ conv2d_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ add[0][0]         │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ digits (Dense)      │ (None, 10)        │      1,290 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 447,370 (1.71 MB)

 Trainable params: 447,370 (1.71 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 30s 30ms/step - accuracy: 0.6634 - loss: 0.9769 - val_accuracy: 0.9121 - val_loss: 0.2930
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 26s 28ms/step - accuracy: 0.9300 - loss: 0.2352 - val_accuracy: 0.9321 - val_loss: 0.2132
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 25s 27ms/step - accuracy: 0.9615 - loss: 0.1274 - val_accuracy: 0.9782 - val_loss: 0.0692
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 25s 27ms/step - accuracy: 0.9728 - loss: 0.0908 - val_accuracy: 0.9303 - val_loss: 0.2158
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 26s 27ms/step - accuracy: 0.9785 - loss: 0.0716 - val_accuracy: 0.9808 - val_loss: 0.0575
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9808 - loss: 0.0575
Test Loss: 0.0575
Test Accuracy: 0.9808


In [13]:
### fixing the size in case of pooling

### Implementation of a residual connection

x = layers.Conv2D(128, (5, 5), activation='relu', padding="same")(mnist_input)
y = layers.Conv2D(128, 3, activation='relu', padding='same')(x) # Applies a transformation to x
y = layers.Conv2D(128, 3, activation='relu', padding='same')(y)
y = layers.MaxPooling2D(2, strides=2)(y)

residual = layers.Conv2D(128, 1, strides=2, padding='same')(x)

y = layers.add([y, residual]) # Adds the original x back to the output features

# Add a classification head for MNIST
output = layers.GlobalAveragePooling2D()(y)
mnist_output = layers.Dense(10, activation='softmax', name='digits')(output)

# Define the model
model = Model(mnist_input, mnist_output)

model.summary()

# --- Load and preprocess actual MNIST data ---
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Reshape and normalize images
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1)).astype('float32') / 255
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1)).astype('float32') / 255

# One-hot encode labels
y_train_one_hot = to_categorical(y_train, num_classes=10)
y_test_one_hot = to_categorical(y_test, num_classes=10)

# Compile the model
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model with actual data
model.fit(x_train, y_train_one_hot, epochs=5, batch_size=64, validation_data=(x_test, y_test_one_hot))

# Evaluate the model on the test set
loss, accuracy = model.evaluate(x_test, y_test_one_hot)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mnist_image         │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 28, 28,    │      3,328 │ mnist_image[0][0] │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 28, 28,    │    147,584 │ conv2d_12[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_14 (Conv2D)  │ (None, 28, 28,    │    147,584 │ conv2d_13[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 14, 14,    │          0 │ conv2d_14[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 14, 14,    │     16,512 │ conv2d_12[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 14, 14,    │          0 │ max_pooling2d[0]… │
│                     │ 128)              │            │ conv2d_15[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ add_1[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ digits (Dense)      │ (None, 10)        │      1,290 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 316,298 (1.21 MB)

 Trainable params: 316,298 (1.21 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 25s 24ms/step - accuracy: 0.6534 - loss: 1.0237 - val_accuracy: 0.8867 - val_loss: 0.3801
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - accuracy: 0.9110 - loss: 0.3002 - val_accuracy: 0.9006 - val_loss: 0.3075
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 20s 21ms/step - accuracy: 0.9500 - loss: 0.1691 - val_accuracy: 0.9640 - val_loss: 0.1208
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 20s 21ms/step - accuracy: 0.9650 - loss: 0.1199 - val_accuracy: 0.9685 - val_loss: 0.0974
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 20s 21ms/step - accuracy: 0.9715 - loss: 0.0966 - val_accuracy: 0.9801 - val_loss: 0.0700
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9801 - loss: 0.0700
Test Loss: 0.0700
Test Accuracy: 0.9801


#### **Layer Weight Sharing**

Consider a model that attempts to assess the semantic similarity between two sentences. It wouldn't make sense to learn two independent models for processing each input sentence. Rather we want to process both with a single LSTM layer. The representations of this  LSTM layer (its weights) are learned based on both inputs simultaneuosly. This is what we called a Siamese LSTM model or a shared LSTM.

In [17]:
### Shared LSTM

from keras import layers
from keras import Input
from keras.models import Model

lstm = layers.LSTM(32) # Instantiate a single LSTM layer once.

# building the left branch of the model
left_input = Input(shape=(None, 128))
left_output = lstm(left_input)

# building the right branch of the model
right_input = Input(shape=(None, 128))
right_output = lstm(right_input)

# build classifier on top
merged = layers.concatenate([left_output, right_output], axis =-1)
predictions = layers.Dense(1, activation='sigmoid')(merged)

# Instantiating the model with two inputs and one output.
model = Model([left_input, right_input], predictions)
model.summary()

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, None, 128) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_6       │ (None, None, 128) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 32)        │     20,608 │ input_layer_5[0]… │
│                     │                   │            │ input_layer_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 64)        │          0 │ lstm_3[0][0],     │
│ (Concatenate)       │                   │            │ lstm_3[1][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │         65 │ concatenate_3[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,673 (80.75 KB)

 Trainable params: 20,673 (80.75 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
import numpy as np
from keras.datasets import imdb
from keras.preprocessing.sequence import pad_sequences

# --- Data Loading and Preprocessing for IMDB ---

max_features = 10000  # Only consider the top 10k words
maxlen = 20         # Cut reviews after 200 words

print("Loading data...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

print(len(x_train), 'train sequences')
print(len(x_test), 'test sequences')

print('Pad sequences (samples x time)')
x_train_padded = pad_sequences(x_train, maxlen=maxlen)
x_test_padded = pad_sequences(x_test, maxlen=maxlen)
print('x_train shape:', x_train_padded.shape)
print('x_test shape:', x_test_padded.shape)

# --- Create an Embedding layer outside the model graph ---

embedding_dim = 128 # This matches the input shape expected by the LSTM model

# This embedding layer will be used to transform integer sequences into dense vectors.
# It's not part of the `model` object directly, but used for data preparation.
embedding_layer = layers.Embedding(max_features, embedding_dim, input_length=maxlen)

print("Generating embedded data...")
# Apply the embedding layer to the padded sequences
# Note: This step might take a moment as it involves forward pass through the embedding layer
embedded_x_train = embedding_layer(x_train_padded).numpy()
embedded_x_test = embedding_layer(x_test_padded).numpy()

# --- Define left_data, right_data, and targets ---
# For a Siamese network, we often feed the same data to both branches for initial training
# or use pairs of similar/dissimilar items.
# For this sentiment analysis example, we'll use the same review for both inputs.

left_data = embedded_x_train
right_data = embedded_x_train
targets = y_train

print("Data preparation complete. Shapes:")
print(f"left_data shape: {left_data.shape}")
print(f"right_data shape: {right_data.shape}")
print(f"targets shape: {targets.shape}")

Loading data...
25000 train sequences
25000 test sequences
Pad sequences (samples x time)
x_train shape: (25000, 20)
x_test shape: (25000, 20)
Generating embedded data...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Data preparation complete. Shapes:
left_data shape: (25000, 20, 128)
right_data shape: (25000, 20, 128)
targets shape: (25000,)


In [19]:
# Configure the model for training
model.compile(optimizer='rmsprop',
              loss='binary_crossentropy',
              metrics=['acc'])

# Train the model
print("\nTraining the shared LSTM model...")
model.fit([left_data, right_data], targets,
          epochs=5,
          batch_size=128,
          validation_split=0.2) # Use a validation split for monitoring

# Evaluate the model on the test set
print("\nEvaluating the model on the test set...")
# For evaluation, we also use the embedded test data for both inputs
test_loss, test_accuracy = model.evaluate([embedded_x_test, embedded_x_test], y_test)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


Training the shared LSTM model...
Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - acc: 0.5246 - loss: 0.6921 - val_acc: 0.5716 - val_loss: 0.6900
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - acc: 0.5835 - loss: 0.6763 - val_acc: 0.5992 - val_loss: 0.6646
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - acc: 0.6083 - loss: 0.6575 - val_acc: 0.5976 - val_loss: 0.6672
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - acc: 0.6216 - loss: 0.6482 - val_acc: 0.6122 - val_loss: 0.6509
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - acc: 0.6291 - loss: 0.6426 - val_acc: 0.5988 - val_loss: 0.6610

Evaluating the model on the test set...
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - acc: 0.6065 - loss: 0.6559
Test Loss: 0.6559
Test Accuracy: 0.6065


So a layer instance may be used more than once -- it can be called arbitrarily many times, reusing the same set of weights every time.

#### **Models as layers**

The functional API, models can be used as we use layers -- effectively. We can think of a model as a "bigger layer". When we call a model instance, we are reusing the weights of the model -- exactly  what happens when we call a layer instance. Calling an instance, whether it's a layer instance or a model instance, will always resuse the existing learned representations of the instance.

In [31]:
### Model as Layers
from keras import layers
from keras import applications
from keras import Input
from keras.models import Model

# Instantiate the Xception base model (weights=None for random initialization, include_top=False to remove the classification head)
xception_base = applications.Xception(weights=None, include_top=False)

# Define the two input branches, each expecting a 250x250x3 image
left_input = Input(shape=(28,28,3), name='left_image')
right_input = Input(shape=(28,28,3), name='right_image')

# Process each input through the shared Xception base model
left_features = xception_base(left_input)
right_features = xception_base(right_input) # Corrected variable name

# Concatenate the features from both branches
merged_features = layers.concatenate([left_features, right_features], axis=-1)

# Add a simple classification head (e.g., for binary classification of similarity)
x = layers.GlobalAveragePooling2D()(merged_features)
output_tensor = layers.Dense(1, activation='sigmoid', name='similarity_output')(x)

# Create the model
model = Model(inputs=[left_input, right_input], outputs=output_tensor)

model.summary()

Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ left_image          │ (None, 28, 28, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ right_image         │ (None, 28, 28, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ xception            │ (None, 1, 1,      │ 20,861,480 │ left_image[0][0], │
│ (Functional)        │ 2048)             │            │ right_image[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 1, 1,      │          0 │ xception[0][0],   │
│ (Concatenate)       │ 4096)             │            │ xception[1][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 4096)      │          0 │ concatenate_6[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ similarity_output   │ (None, 1)         │      4,097 │ global_average_p… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,865,577 (79.60 MB)

 Trainable params: 20,811,049 (79.39 MB)

 Non-trainable params: 54,528 (213.00 KB)

In [32]:
import tensorflow as tf
import numpy as np
from keras.datasets import mnist # Changed to MNIST

# 1. Load MNIST dataset
print("Loading MNIST dataset...")
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# MNIST images are 28x28 grayscale. Model expects 250x250x3.
# We will resize and convert to 3 channels.

# 2. Preprocess images: Resize to 250x250 and normalize to [0, 1] and convert to 3 channels
#target_size = (28, 28)

def preprocess_image_mnist(image):
    # MNIST images are 28x28, need to expand to 3 channels and resize
    image = tf.expand_dims(image, axis=-1) # Add channel dimension (28, 28, 1)
    image = tf.image.grayscale_to_rgb(image) # Convert to 3 channels (28, 28, 3)
    #image = tf.image.resize(image, target_size) # Resize to target_size (250, 250, 3)
    image = tf.cast(image, tf.float32) / 255.0
    return image

print(f"Resizing and preprocessing {len(x_train)} training images to {target_size} (3 channels)...")
x_train_resized = np.array([preprocess_image_mnist(img) for img in x_train])
print(f"Resizing and preprocessing {len(x_test)} test images to {target_size} (3 channels)...")
x_test_resized = np.array([preprocess_image_mnist(img) for img in x_test])

# Flatten y_train and y_test for easier class grouping
y_train = y_train.flatten()
y_test = y_test.flatten()

# Group images by class for easier pair generation
train_data_by_class = {i: x_train_resized[y_train == i] for i in range(10)}
test_data_by_class = {i: x_test_resized[y_test == i] for i in range(10)}

# 3. Function to generate image pairs for a binary similarity task
def generate_pairs(data_by_class, num_pairs):
    left_images = []
    right_images = []
    labels = [] # 1 for similar pair (same class), 0 for dissimilar pair (different classes)
    num_classes = len(data_by_class)

    for _ in range(num_pairs):
        # 50% chance to generate a similar pair, 50% for a dissimilar pair
        if np.random.rand() < 0.5:
            # Generate similar pair
            class_idx = np.random.randint(num_classes)
            # Ensure there are at least two images in the class to pick from
            if len(data_by_class[class_idx]) < 2:
                continue # Skip if not enough images in class
            idx1, idx2 = np.random.choice(len(data_by_class[class_idx]), 2, replace=False)
            left_images.append(data_by_class[class_idx][idx1])
            right_images.append(data_by_class[class_idx][idx2])
            labels.append(1)
        else:
            # Generate dissimilar pair
            class_idx1, class_idx2 = np.random.choice(num_classes, 2, replace=False)
            # Ensure there's at least one image in each chosen class
            if len(data_by_class[class_idx1]) < 1 or len(data_by_class[class_idx2]) < 1:
                continue # Skip if not enough images
            idx1 = np.random.randint(len(data_by_class[class_idx1]))
            idx2 = np.random.randint(len(data_by_class[class_idx2]))
            left_images.append(data_by_class[class_idx1][idx1])
            right_images.append(data_by_class[class_idx2][idx2])
            labels.append(0)

    return np.array(left_images), np.array(right_images), np.array(labels)

# Generate a subset of training and testing pairs for demonstration
# Adjust num_train_pairs and num_test_pairs based on computational resources
num_train_pairs = 10000
num_test_pairs = 2000

print(f"Generating {num_train_pairs} training pairs...")
train_left_data, train_right_data, train_targets = generate_pairs(train_data_by_class, num_train_pairs)
print(f"Generating {num_test_pairs} test pairs...")
test_left_data, test_right_data, test_targets = generate_pairs(test_data_by_class, num_test_pairs)

print(f"Train data shapes: left={train_left_data.shape}, right={train_right_data.shape}, targets={train_targets.shape}")
print(f"Test data shapes: left={test_left_data.shape}, right={test_right_data.shape}, targets={test_targets.shape}")

# 4. Compile the model (already defined as 'model' in Vx4CKfjrcmNQ) and train

print("\nCompiling the model...")
model.compile(optimizer='adam', # Using Adam optimizer as a common choice
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("\nTraining the model...")
# Train the model with the generated pairs
history = model.fit([train_left_data, train_right_data], train_targets,
                    epochs=3, # Reduced epochs for a quicker demonstration
                    batch_size=32,
                    validation_data=([test_left_data, test_right_data], test_targets))

print("\nEvaluation on test data...")
loss, accuracy = model.evaluate([test_left_data, test_right_data], test_targets)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Loading MNIST dataset...
Resizing and preprocessing 60000 training images to (250, 250) (3 channels)...
Resizing and preprocessing 10000 test images to (250, 250) (3 channels)...
Generating 10000 training pairs...
Generating 2000 test pairs...
Train data shapes: left=(10000, 28, 28, 3), right=(10000, 28, 28, 3), targets=(10000,)
Test data shapes: left=(2000, 28, 28, 3), right=(2000, 28, 28, 3), targets=(2000,)

Compiling the model...

Training the model...
Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 122s 180ms/step - accuracy: 0.5033 - loss: 0.7008 - val_accuracy: 0.4855 - val_loss: 0.6932
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 43ms/step - accuracy: 0.4939 - loss: 0.6934 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 43ms/step - accuracy: 0.4987 - loss: 0.6935 - val_accuracy: 0.5145 - val_loss: 0.6931

Evaluation on test data...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5145 - loss: 0.6931
Test Loss: 0.6931
Test Accuracy: 0.5145


That's it. We not just learned how to implement complex network, we also apply them on real dataset.